# CF Modeling — Pycortex Flatmap Visualisation

Visualises the outputs of the CF modeling pipeline (`02_fit_cf_model.py`) using pycortex flatmaps.

**Pipeline outputs** (grayordinate space, 59 412 vertices):
- `R2_full` — full banded-ridge model R²
- `R2_{ROI_A}` / `R2_{ROI_B}` — per-band split R² (raw)
- `R2_{ROI_A}_nc` / `R2_{ROI_B}_nc` — null-corrected split R²
- `R2_null_{ROI_A}` / `R2_null_{ROI_B}` — OLS null-model R²
- `Shared_R2` — shared variance (R²_A + R²_B − R²_full)
- `product_map` — integration score: √(clip(R²_A_nc,0) × clip(R²_B_nc,0))

**Pycortex note**: `cortex.Vertex` expects a **64 984-vertex bilateral array** (32 492 per hemisphere,
medial wall included). The grayordinate→vertex conversion fills zeros at medial-wall positions.

In [ ]:
# =============================================================================
# Cell 0 — Imports and pycortex setup
# =============================================================================
# IMPORTANT: set PYCORTEX_FILESTORE environment variable BEFORE importing cortex,
# otherwise pycortex caches the wrong filestore path on first import.

import sys
import os

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# --- Pycortex filestore (must precede `import cortex`) ---
PYCORTEX_STORE = "/home/amin/Research/Representation/Movie/data/hedger2026"
os.environ["PYCORTEX_FILESTORE"] = PYCORTEX_STORE

import cortex

# Override the in-process singleton as well, matching the pattern used in viz.ipynb.
new_db = cortex.database.Database(PYCORTEX_STORE)
cortex.db = new_db
cortex.database.default_filestore = new_db
for _mod_name, _mod in sys.modules.items():
    if _mod_name.startswith("cortex") and hasattr(_mod, "db"):
        _mod.db = new_db
cortex.options.config.set("basic", "filestore", PYCORTEX_STORE)

# --- Vicsompy visualisation helpers ---
VICSOMPY_REPO = "/home/amin/Research/Representation/Movie/Vicarious_somatotopy"
if VICSOMPY_REPO not in sys.path:
    sys.path.insert(0, VICSOMPY_REPO)

from vicsompy.vis import basic_plot, alpha_plot, Plot

print("Imports complete.")
print("Pycortex subjects available:", list(cortex.db.subjects.keys()))

In [ ]:
# =============================================================================
# Cell 1 — CONFIG  (edit these paths/names before running)
# =============================================================================

# --- ROI pair and analysis mode ---
ROI_A = "3b"           # somatosensory seed ROI (Glasser short name)
ROI_B = "V1"           # visual seed ROI (Glasser short name)
MODE  = "group_average"  # "group_average" or "per_subject"

# --- Paths ---
OUTPUT_BASE = "/home/amin/Research/Representation/Movie/outputs/cf_modeling"

# HCP 32k_fs_LR curvature CIFTI — used as BrainModelAxis template.
# Shape: (1, 59 412); vertices L=29 696, R=29 716.
TEMPLATE_CIFTI = (
    "/home/amin/Research/Representation/Movie/data/"
    "HCP_S1200_GroupAvg_v1/S1200.curvature_MSMAll.32k_fs_LR.dscalar.nii"
)

# Pycortex subject name registered in the filestore above.
SUBJECT = "hcp_999999_draw_NH"

# --- Output figures directory ---
FIGS_DIR = os.path.join(OUTPUT_BASE, MODE, f"{ROI_A}_{ROI_B}", "figures")
os.makedirs(FIGS_DIR, exist_ok=True)

print(f"ROI pair  : {ROI_A} × {ROI_B}")
print(f"Mode      : {MODE}")
print(f"Figures   : {FIGS_DIR}")

In [ ]:
# =============================================================================
# Cell 2 — Load R² maps from npy
# =============================================================================
# Map names produced by cf_model.py → save_all_maps():
#   R2_full, R2_{roi_a}, R2_{roi_b}, Shared_R2,
#   R2_null_{roi_a}, R2_null_{roi_b},
#   R2_{roi_a}_nc, R2_{roi_b}_nc, product_map

if MODE == "group_average":
    maps_dir = os.path.join(OUTPUT_BASE, MODE, f"{ROI_A}_{ROI_B}", "prep")
else:
    # Per-subject mode: use group statistics averaged across subjects.
    maps_dir = os.path.join(OUTPUT_BASE, MODE, f"{ROI_A}_{ROI_B}", "group", "stats")

print(f"Loading maps from: {maps_dir}")


def load_map(name: str) -> np.ndarray | None:
    """Load a grayordinate R² map from disk.

    Returns a float32 array of shape (59 412,) or None if the file is missing.
    """
    path = os.path.join(maps_dir, f"{name}.npy")
    if not os.path.exists(path):
        print(f"  WARNING: not found — {path}")
        return None
    arr = np.load(path).astype(np.float32)
    print(f"  {name:<30s}  shape={arr.shape}  mean={np.nanmean(arr):+.4f}  max={np.nanmax(arr):.4f}")
    return arr


R2_full   = load_map("R2_full")
R2_a      = load_map(f"R2_{ROI_A}")
R2_b      = load_map(f"R2_{ROI_B}")
R2_a_nc   = load_map(f"R2_{ROI_A}_nc")
R2_b_nc   = load_map(f"R2_{ROI_B}_nc")
Shared_R2 = load_map("Shared_R2")
product   = load_map("product_map")
R2_null_a = load_map(f"R2_null_{ROI_A}")
R2_null_b = load_map(f"R2_null_{ROI_B}")

_first = next((v for v in [R2_full, R2_a, R2_b] if v is not None), None)
print("\nMaps loaded. Grayordinate shape:", _first.shape if _first is not None else "N/A")

In [ ]:
# =============================================================================
# Cell 3 — Grayordinate → pycortex vertex conversion
# =============================================================================
# pycortex cortex.Vertex expects a (64 984,) bilateral vertex array where:
#   - Left hemisphere  : indices 0 … 32 491  (32 492 vertices, medial wall = 0)
#   - Right hemisphere : indices 32 492 … 64 983
#
# The template CIFTI BrainModelAxis maps grayordinate rows to vertex indices
# within each hemisphere's 32 492-vertex 32k_fs_LR surface.

N_VERTS_PER_HEM = 32_492   # HCP 32k_fs_LR surface vertices per hemisphere

# Cache the BrainModelAxis once (loading the CIFTI header is fast).
_template_img = nib.load(TEMPLATE_CIFTI)
_bm_axis = _template_img.header.get_axis(1)
print(f"Template CIFTI shape : {_template_img.shape}")
print(f"BrainModelAxis type  : {type(_bm_axis).__name__}")


def grayord_to_vertex(
    grayord_data: np.ndarray,
    bm_axis=_bm_axis,
    n_verts_per_hem: int = N_VERTS_PER_HEM,
) -> np.ndarray:
    """Convert a (59 412,) grayordinate array to a (64 984,) bilateral vertex array.

    Medial-wall vertices (not present in the grayordinate space) are filled with
    zero so that pycortex renders them as the cortical background.

    Parameters
    ----------
    grayord_data    : (N_GRAYORD,) float array — CIFTI grayordinate values.
    bm_axis         : nibabel BrainModelAxis — from the template CIFTI header.
    n_verts_per_hem : int — vertices per hemisphere in the surface mesh (32 492).

    Returns
    -------
    bilateral : (2 * n_verts_per_hem,) float32
        Left hemisphere in [0, n_verts_per_hem),
        right hemisphere in [n_verts_per_hem, 2*n_verts_per_hem).
    """
    bilateral = np.zeros(2 * n_verts_per_hem, dtype=np.float32)
    pos = 0
    for name, _, model in bm_axis.iter_structures():
        n = len(model.vertex)
        if "CORTEX_LEFT" in name:
            bilateral[model.vertex] = grayord_data[pos : pos + n]
        elif "CORTEX_RIGHT" in name:
            bilateral[n_verts_per_hem + model.vertex] = grayord_data[pos : pos + n]
        else:
            # Non-cortical structures (subcortical, cerebellum) — skip.
            pos += n
            continue
        pos += n
    return bilateral


def to_pcx(arr: np.ndarray | None) -> np.ndarray | None:
    """Convenience wrapper: grayord → pycortex bilateral, or None if input is None."""
    return None if arr is None else grayord_to_vertex(arr)


# Convert every map to pycortex vertex space.
pcx = {
    "R2_full":              to_pcx(R2_full),
    f"R2_{ROI_A}":          to_pcx(R2_a),
    f"R2_{ROI_B}":          to_pcx(R2_b),
    f"R2_{ROI_A}_nc":       to_pcx(R2_a_nc),
    f"R2_{ROI_B}_nc":       to_pcx(R2_b_nc),
    "Shared_R2":            to_pcx(Shared_R2),
    "product_map":          to_pcx(product),
    f"R2_null_{ROI_A}":     to_pcx(R2_null_a),
    f"R2_null_{ROI_B}":     to_pcx(R2_null_b),
}

_bilateral_shape = next((v.shape for v in pcx.values() if v is not None), None)
print(f"Vertex maps ready. Bilateral shape: {_bilateral_shape}")
print(f"  L vertices : {N_VERTS_PER_HEM}  (indices 0 … {N_VERTS_PER_HEM - 1})")
print(f"  R vertices : {N_VERTS_PER_HEM}  (indices {N_VERTS_PER_HEM} … {2 * N_VERTS_PER_HEM - 1})")

In [ ]:
# =============================================================================
# Cell 4 — 1-D flatmaps grid  (single colorbar per map, basic_plot)
# =============================================================================
# Each panel is rendered with cortex.Vertex + cortex.quickshow via basic_plot().
# Colourmap / vmin / vmax are tuned per map type.

MAP_CONFIGS = [
    # (pcx_key,              panel title,                       cmap,       vmin,  vmax)
    (f"R2_{ROI_A}_nc",  f"R² {ROI_A} null-corrected",  "plasma",   0.0,   0.30),
    (f"R2_{ROI_B}_nc",  f"R² {ROI_B} null-corrected",  "plasma",   0.0,   0.30),
    ("R2_full",          "R² full model",                 "plasma",   0.0,   0.30),
    ("Shared_R2",         "Shared R²",                    "RdBu_r",  -0.10,  0.10),
    ("product_map",       "Integration score",            "plasma",   0.0,   0.15),
    (f"R2_{ROI_A}",     f"R² {ROI_A} (raw split)",      "plasma",   0.0,   0.30),
    (f"R2_{ROI_B}",     f"R² {ROI_B} (raw split)",      "plasma",   0.0,   0.30),
    (f"R2_null_{ROI_A}", f"R² null {ROI_A}",             "Greys",    0.0,   0.10),
    (f"R2_null_{ROI_B}", f"R² null {ROI_B}",             "Greys",    0.0,   0.10),
]

n_cols = 4
n_rows = (len(MAP_CONFIGS) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = axes.flatten()

for i, (key, title, cmap, vmin, vmax) in enumerate(MAP_CONFIGS):
    ax = axes[i]
    dat = pcx.get(key)
    if dat is None:
        ax.set_visible(False)
        continue
    basic_plot(
        dat, vmax=vmax, vmin=vmin,
        subject=SUBJECT, cmap=cmap,
        ax=ax, colorbar=True, labels=True,
    )
    ax.set_title(title, fontsize=12)

# Hide unused panels.
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f"CF Modeling: {ROI_A} × {ROI_B}   ({MODE})", fontsize=15)
plt.tight_layout()

out_path = os.path.join(FIGS_DIR, "1d_flatmaps.png")
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

In [ ]:
# =============================================================================
# Cell 5 — 2-D flatmap: ROI_A vs ROI_B  (Hedger Figure 3a style)
# =============================================================================
# Uses cortex.Vertex2D via Plot.uber_plot().
#   dat  = R²_A_nc  → dimension 1 (hue in the 2-D colour wheel)
#   dat2 = R²_B_nc  → dimension 2 (alpha / saturation in the 2-D colour wheel)
#
# IMPORTANT: uber_plot() checks `if 'dat2' not in kwargs` to decide between
# the 1-D and 2-D branches.  We MUST pass dat2 as a keyword argument to
# uber_plot() — even though it was also supplied to Plot() — so that the
# 2-D Vertex2D branch is taken.  Omitting it silently falls back to the 1-D
# branch (dat2 = ones_like(dat)), producing a plain alpha-scaled single map.

dat_a = pcx.get(f"R2_{ROI_A}_nc")
dat_b = pcx.get(f"R2_{ROI_B}_nc")

if dat_a is not None and dat_b is not None:
    plotter_2d = Plot(
        dat=dat_a,
        dat2=dat_b,
        subject=SUBJECT,
        vmin=0.0,   vmax=0.3,
        vmin2=0.0,  vmax2=0.3,
        cmap="nipy_spectral_alpha",
        x=14, y=7,
        dpi=150,
        with_curvature=True,
        with_colorbar=True,
        with_labels=True,
    )
    # Pass dat2 explicitly so uber_plot() enters the 2-D Vertex2D branch.
    fig_2d = plotter_2d.uber_plot(dat2=dat_b)
    fig_2d.suptitle(
        f"R² {ROI_A} nc (dim 1) × R² {ROI_B} nc (dim 2)   [{MODE}]",
        fontsize=13,
    )
    out_2d = os.path.join(FIGS_DIR, f"2d_flatmap_{ROI_A}_x_{ROI_B}.png")
    plotter_2d.saveout(out_2d)
    print(f"Saved: {out_2d}")
    plt.show()
else:
    print("Skipped 2-D flatmap: one or both null-corrected maps are missing.")

In [ ]:
# =============================================================================
# Cell 6 — 2-D flatmap: integration score overlaid on modality balance
# =============================================================================
# dat  = modality_balance = R²_A_nc − R²_B_nc
#           positive → ROI_A dominant, negative → ROI_B dominant
# dat2 = product_map (integration score: geometric mean of both nc maps)
#           high → vertex responds to both modalities
#
# The alpha dimension (dat2) acts as a mask: unimodal vertices (low product)
# become transparent, while bimodal vertices (high product) are opaque.
#
# IMPORTANT: same uber_plot() 2-D branch requirement as Cell 5 — dat2 must
# be passed explicitly to uber_plot(), not just to Plot().

prod = pcx.get("product_map")

if dat_a is not None and dat_b is not None and prod is not None:
    modality_balance = dat_a - dat_b   # (64 984,) float32

    plotter_int = Plot(
        dat=modality_balance,
        dat2=prod,
        subject=SUBJECT,
        vmin=-0.3,  vmax=0.3,    # balance: symmetric around zero
        vmin2=0.0,  vmax2=0.15,  # product: non-negative
        cmap="nipy_spectral_alpha",
        x=14, y=7,
        dpi=150,
        with_curvature=True,
        with_colorbar=True,
        with_labels=True,
    )
    # Pass dat2 explicitly so uber_plot() enters the 2-D Vertex2D branch.
    fig_int = plotter_int.uber_plot(dat2=prod)
    fig_int.suptitle(
        f"Modality balance ({ROI_A}−{ROI_B}) coloured, integration score (alpha)   [{MODE}]",
        fontsize=12,
    )
    out_int = os.path.join(FIGS_DIR, "2d_integration_modality_balance.png")
    plotter_int.saveout(out_int)
    print(f"Saved: {out_int}")
    plt.show()
else:
    print("Skipped integration map: one or more required maps are missing.")

In [ ]:
# =============================================================================
# Cell 7 — Save all maps as a single multi-map CIFTI dscalar
# =============================================================================
# Re-saves the npy maps as a combined CIFTI for quick inspection in wb_view.
# (02_fit_cf_model.py already does this during fitting; this cell lets you
# re-generate the CIFTI independently, e.g. after editing vmin/vmax or
# switching between group_average and per_subject.)

template_img = nib.load(TEMPLATE_CIFTI)
bm_axis_out  = template_img.header.get_axis(1)

# Collect available grayordinate maps (original shape, not vertex-inflated).
all_maps_gray = {
    f"R2_{ROI_A}_nc":    R2_a_nc,
    f"R2_{ROI_B}_nc":    R2_b_nc,
    "R2_full":           R2_full,
    "Shared_R2":         Shared_R2,
    "product_map":       product,
    f"R2_{ROI_A}":       R2_a,
    f"R2_{ROI_B}":       R2_b,
    f"R2_null_{ROI_A}": R2_null_a,
    f"R2_null_{ROI_B}": R2_null_b,
}
# Drop any map that was not found on disk.
all_maps_gray = {k: v for k, v in all_maps_gray.items() if v is not None}

map_names = list(all_maps_gray.keys())
data_2d   = np.stack(
    [all_maps_gray[k].astype(np.float32) for k in map_names], axis=0
)  # (n_maps, 59 412)

scalar_axis = nib.cifti2.ScalarAxis(map_names)
header      = nib.cifti2.Cifti2Header.from_axes((scalar_axis, bm_axis_out))
img_out     = nib.Cifti2Image(data_2d, header=header)

cifti_maps_dir = os.path.join(OUTPUT_BASE, MODE, f"{ROI_A}_{ROI_B}", "cifti_maps")
os.makedirs(cifti_maps_dir, exist_ok=True)

out_cifti = os.path.join(cifti_maps_dir, "all_maps_viz.dscalar.nii")
nib.save(img_out, out_cifti)

print(f"Saved multimap CIFTI ({len(map_names)} maps): {out_cifti}")
print("Maps included:", map_names)
print()
print("Open in wb_view:")
print(f"  wb_view {TEMPLATE_CIFTI} {out_cifti}")

In [ ]:
# =============================================================================
# Cell 8 — Summary statistics table
# =============================================================================
# Reports mean R², fraction of vertices with positive R², and maximum value
# across all 59 412 grayordinate vertices for each output map.

_stats_maps = [
    (f"R2_{ROI_A}",       R2_a),
    (f"R2_{ROI_B}",       R2_b),
    (f"R2_{ROI_A}_nc",    R2_a_nc),
    (f"R2_{ROI_B}_nc",    R2_b_nc),
    ("R2_full",           R2_full),
    ("Shared_R2",         Shared_R2),
    (f"R2_null_{ROI_A}", R2_null_a),
    (f"R2_null_{ROI_B}", R2_null_b),
    ("product_map",       product),
]

print(f"{'Map':<30}  {'Mean R²':>10}  {'frac>0':>9}  {'Max':>10}")
print("-" * 65)
for name, arr in _stats_maps:
    if arr is None:
        print(f"{name:<30}  {'—':>10}  {'—':>9}  {'—':>10}")
        continue
    print(
        f"{name:<30}"
        f"  {np.nanmean(arr):>+10.4f}"
        f"  {np.mean(arr > 0) * 100:>8.1f}%"
        f"  {np.nanmax(arr):>10.4f}"
    )